# Task 3: Thiết kế Kafka Topic (1.5 điểm)

## 1. Phương pháp luận và Lý do thiết kế

Hệ thống sử dụng **Apache Kafka** làm Message Broker trung tâm. Theo yêu cầu, chúng tôi đã phân rã luồng dữ liệu thành 4 topics riêng biệt thay vì dồn chung vào một topic. Lý do thiết kế như sau:

1. **`node_events`**: Chứa thông tin về các AST node. Việc tách riêng giúp Consumer phía Neo4j chỉ việc tạo Node (không bận tâm đến Edge), tăng tốc độ ghi.
2. **`edge_events`**: Chứa thông tin về liên kết (CFG, DFG, Call). Neo4j Sink Connector sẽ đợi Node được tạo rồi mới tiến hành nối các Edge này một cách độc lập.
3. **`source_metadata_events`**: Chứa siêu dữ liệu của file (hash, kích thước, LOC). Dữ liệu này có bản chất khác biệt (Document-oriented) nên được tách ra để Apache Spark dễ dàng tiêu thụ và đẩy thẳng vào MongoDB mà không bị nhiễu bởi dữ liệu đồ thị.
4. **`parser_error_events`**: Dành riêng cho việc theo dõi file lỗi, phục vụ cho quá trình giám sát và gỡ lỗi (Monitoring/Debugging) mà không làm gián đoạn luồng dữ liệu chính.

**Đảm bảo tính Tương thích và Dấu vết thời gian (Schema & Event Time):**
Tất cả các message được chuẩn hóa dưới dạng JSON Schema. Bên trong mỗi JSON payload, chúng tôi luôn đính kèm 2 trường bắt buộc:
- `"schema_version"`: Cho phép tiến hóa cấu trúc dữ liệu trong tương lai (Forward Compatibility).
- `"event_time"`: Ghi nhận thời gian chính xác sự kiện được tạo ra (Event-time processing) thay vì thời gian đến Kafka (Processing-time). Điều này đặc biệt quan trọng khi chạy Spark Structured Streaming với các kỹ thuật như Watermarking.


## 2. Giao diện quản trị Kafka (Kafka UI)

Dưới đây là hình ảnh minh chứng 4 topics đã được tạo thành công trên cụm Kafka.



![Kafka UI Topics](images/task3_1_1.png)

In [ ]:
# Ô Notebook thực thi: Lấy mẫu thử một Kafka Message để kiểm tra Schema
# YÊU CẦU: HÃY CHẠY Ô CODE NÀY TRONG VS CODE HOẶC JUPYTER ĐỂ NÓ HIỂN THỊ KẾT QUẢ BÊN DƯỚI!
from kafka import KafkaConsumer
import json

# Khởi tạo Consumer để đọc thử 1 tin nhắn từ topic node_events
consumer = KafkaConsumer(
    'node_events',
    bootstrap_servers=['127.0.0.1:9092'],
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=5000
)

# Lấy chính xác 1 message đầu tiên và dừng lại
for msg in consumer:
    print("🎯 Đã nhận được 1 mẫu tin nhắn từ Topic 'node_events':\n")
    print(json.dumps(msg.value, indent=4, ensure_ascii=False))
    break
    
consumer.close()



## 3. Reflection (Phản ngẫm của Tý)

**Những gì hiệu quả:**
- Việc tách riêng 4 topic giúp phân tách trách nhiệm (Separation of Concerns) rất tốt. Luồng Neo4j chỉ việc tập trung vào Node/Edge, luồng Spark chỉ tập trung vào Metadata. Quá trình vận hành trơn tru và không bị thắt cổ chai.
- Việc áp đặt Schema Version và định dạng JSON tĩnh ngay từ đầu giúp hệ thống tránh được các rác dữ liệu.

**Những gì gặp khó khăn & Cách giải quyết:**
- Khó khăn ban đầu: Team định dồn chung Node và Edge vào 1 topic để duy trì thứ tự sự kiện cho dễ kiểm soát. Tuy nhiên, khi dùng Neo4j Kafka Connector, nó yêu cầu lệnh Cypher riêng biệt cho Node và Edge, nếu gộp chung sẽ gây quá tải logic ở phía Sink. 
- *Cách giải quyết:* Quyết định tách dứt khoát thành `node_events` và `edge_events`. Nhờ đó, việc viết lệnh Cypher `MERGE` cho Sink trở nên vô cùng ngắn gọn, sạch sẽ và tối ưu hóa hiệu năng chèn (Batch Insert).


![Kafka UI Messages](images/task3_1_2.png)